In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from openpiv import pyprocess, preprocess, validation, filters, piv
import scipy.signal
from scipy.ndimage import gaussian_filter as smoo 
from scipy import ndimage as ndi
from matplotlib.animation import FuncAnimation, writers
from matplotlib import animation
from IPython.display import display, HTML
from matplotlib.animation import PillowWriter
from pathlib import Path
from scipy.ndimage import uniform_filter

In [2]:
# PIVdiv on top of ERA5 clouds 

dsp = xr.open_dataset('/Users/bmapes/Box/GWaves_2023_10_11-14_SEPAC/PIV_outputs/2023-10-12_first_attempts/IR_2023_10_12.PIVoutput_40_30.nc')
dsp

<xarray.Dataset> Size: 37MB
Dimensions:       (time: 47, latitude: 109, longitude: 147)
Coordinates:
  * latitude      (latitude) float32 436B 9.14 8.74 8.34 ... -33.66 -34.06
  * longitude     (longitude) float32 588B -129.1 -128.7 ... -71.14 -70.74
  * time          (time) datetime64[ns] 376B 2023-10-12T00:15:00 ... 2023-10-...
Data variables:
    u             (time, latitude, longitude) float64 6MB ...
    v             (time, latitude, longitude) float64 6MB ...
    sig2noise     (time, latitude, longitude) float64 6MB ...
    u2            (time, latitude, longitude) float64 6MB ...
    v2            (time, latitude, longitude) float64 6MB ...
    invalid_mask  (time, latitude, longitude) bool 753kB ...
    divergence    (time, latitude, longitude) float64 6MB ...
Attributes:
    window_size:       40
    overlap:           30
    search_area_size:  40
    dt_seconds:        1800.0
    source:            OpenPIV extended_search_area_piv
    valid_fraction:    0.45884041690070526

In [3]:
dse = xr.open_dataset('/Users/bmapes/Box/GWaves_2023_10_11-14_SEPAC/2023_Oct11-14_ERA5_2Dfields/data_stream-oper_stepType-instant.nc')
dse

<xarray.Dataset> Size: 113MB
Dimensions:     (valid_time: 96, latitude: 261, longitude: 281)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 768B 2023-10-11 ... 2023-10-14T23...
  * latitude    (latitude) float64 2kB 10.0 9.75 9.5 9.25 ... -54.5 -54.75 -55.0
  * longitude   (longitude) float64 2kB -140.0 -139.8 -139.5 ... -70.25 -70.0
    expver      (valid_time) <U4 2kB ...
Data variables:
    msl         (valid_time, latitude, longitude) float32 28MB ...
    aluvd       (valid_time, latitude, longitude) float32 28MB ...
    lcc         (valid_time, latitude, longitude) float32 28MB ...
    blh         (valid_time, latitude, longitude) float32 28MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2024-12-16T18:40 GRIB to CDM+CF via cfgrib-0.9.1...

In [4]:
# Time=smoothed div on lcc animation 

# lcc 
lcc = dse.lcc.sel(valid_time=slice('2023-10-12-T13','2023-10-12-T22'), latitude=slice(-11, -24), longitude=slice(-102,-88))

# make animation set 
data = [(lcc.isel(valid_time=t)) for t in range(len(lcc.valid_time))]

# Plot title 
titlestring = "E5 low cloud cover, "

fig, ax = plt.subplots()
ani = animation.FuncAnimation(fig, lambda f: (ax.clear(),
                                              data[f].plot(ax=ax, add_colorbar=False, vmin=0, vmax=1, cmap='Greys_r'),
                                              ax.set_title(titlestring + ax.get_title())),
                                              frames=len(data), interval=100)

ani.save('/Users/bmapes/Box/GWaves_2023_10_11-14_SEPAC/PIV_outputs/ERA5lccanimation.gif', writer='pillow')

display(HTML(ani.to_jshtml()))
plt.close()

In [5]:
# Time=smoothed div on lcc animation 

# lcc 
lcc = dse.lcc.sel(valid_time=slice('2023-10-12-T13','2023-10-12-T22'), latitude=slice(-11, -24), longitude=slice(-102,-88))

# make animation set 
data = [(lcc.isel(valid_time=t)) for t in range(len(lcc.valid_time))]

# Plot title 
titlestring = "E5 low cloud cover, "

fig, ax = plt.subplots()
ani = animation.FuncAnimation(fig, lambda f: (ax.clear(),
                                              data[f].plot(ax=ax, add_colorbar=False, vmin=0, vmax=1, cmap='Greys_r'),
                                              ax.set_title(titlestring + ax.get_title())),
                                              frames=len(data), interval=100)

ani.save('/Users/bmapes/Box/GWaves_2023_10_11-14_SEPAC/PIV_outputs/ERA5lccanimation.gif', writer='pillow')

display(HTML(ani.to_jshtml()))
plt.close()

In [6]:
#data = [(lcc.isel(valid_time=t)) for t in range(len(lcc.valid_time))]

# smooth div in time, lat, lon, lightly 
xytsmoodiv = dsp.divergence.rolling(time=3, latitude=2, longitude=2, center=True).mean()
data = [xytsmoodiv.isel(time=t) for t in range(len(xytsmoodiv.time)-3)]

# Data sets for overlay: 
data = [lcc.isel(valid_time=t) for t in range(lcc.sizes['valid_time'])]
div  = [xytsmoodiv.sel(time=t, method='nearest').sel(latitude=slice(-12,-23), longitude=slice(-102,-88)) 
        for t in lcc.valid_time.values]


titlestring = "E5 low cloud + sat. PIV div, "

fig, ax = plt.subplots()
ani = animation.FuncAnimation(fig, lambda f: (ax.clear(),
                                              data[f].plot(ax=ax, add_colorbar=False, vmin=0, vmax=1, cmap='Greys_r'),
                                              div[f].plot.contour(ax=ax, levels=np.linspace(-0.2e-4, 0.2e-4, 8)),
                                              ax.set_title(titlestring + ax.get_title())),
                                              frames=len(data), interval=300)

ani.save('/Users/bmapes/Box/GWaves_2023_10_11-14_SEPAC/PIV_outputs/ERA5lccanimation+PIVdiv.gif', writer='pillow')

display(HTML(ani.to_jshtml()))
plt.close()